# Project Background (Chicago Business License x Parcel )


**Learning Goals**
- Understand project context and data sources
- Import data from OData endpoints
- Build a PIN-based commercial parcel filter
- Perform spatial matching from business points to parcel polygons
- Export clean analysis tables to Parquet

**Notebook Structure**
- Part 1. Preparation and Project Background
- Part 2. Data Import and Dataset Understanding
- Part 3. Combination Workflow (Step 1/2/3)
- Part 4. Scalable Multi-Year Pipeline

# Part 1. Preparation

## 1-1. Preparation: Environment and Python Basics

### A) Python Environment in VS Code
1. Install Python 3.11+ (recommended: 3.12)
2. In project root, create a local virtual environment:
   - Windows PowerShell: `python -m venv .venv`
3. Activate it:
   - PowerShell: `.venv\\Scripts\\Activate.ps1`
4. In VS Code:
   - `Ctrl+Shift+P` -> `Python: Select Interpreter`
   - Choose `.venv`
5. For notebooks:
   - Open this notebook
   - Click the kernel picker (top-right)
   - Select the same `.venv` kernel


### B) ▶️Install Required Packages
Run once in terminal:

```bash
python -m pip install --upgrade pip
pip install pandas geopandas matplotlib pyarrow shapely pyogrio fiona requests jupyter ipykernel
```

Optional:

```bash
pip install tqdm
```

- `requests`: required for the Google Drive inspection / download cells in Part 4
- `tqdm`: optional, only used to show a progress bar for chunked local CSV reading or multi-year runs
- Standard-library modules such as `json`, `time`, `io`, `zipfile`, `re`, `struct`, and `pathlib` are built into Python and do not need separate installation

In [1]:
# # Run this if packages are missing in the selected notebook kernel:
# %pip install --upgrade pip
# %pip install pandas geopandas matplotlib pyarrow shapely pyogrio fiona requests jupyter ipykernel
# # Optional:
# %pip install tqdm

In [2]:
from pathlib import Path

import pandas as pd
import geopandas as gpd


def pick_first_existing(candidates):
    """Return the first existing local path from a candidate list."""
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError(f"None of these files exist: {candidates}")


### C) Super Basic Python Notes
- `import ...` means loading an external library into your script so you can use its functions
- Add comments with `#`
- Multi-line comments are usually written as multiple lines with `#`
- VS Code comment shortcut:
  - `Ctrl + /` (toggle comment for selected lines)

## 1-2. Task Background

Chicago has many business licenses across different years, and each license has location information (latitude/longitude).

We want to:
1. Identify commercial parcels using assessment class information
2. Match business license points to parcel polygons
3. Build clean analysis tables by year
4. Save outputs as Parquet for fast downstream analytics

Core challenge:
- Business license table and parcel polygon table do not share a direct key for a simple join
- We solve this by combining:
  - PIN-Class mapping (from Assessed Values dataset)
  - Spatial join (point-in-polygon)

# ▶️ Part 2. Data Import and Source Overview


1. Chicago Business Licenses (All)  <- Main dataset used in this tutorial
- Portal: https://data.cityofchicago.org/Community-Economic-Development/Business-Licenses/r5kz-chrr/about_data
- OData: https://data.cityofchicago.org/api/odata/v4/r5kz-chrr?$format=json

 <span style="color:#888888;">Current Active (not used in this notebook): data.cityofchicago.org/Community-Economic-Development/Business-Licenses-Current-Active/uupf-x98q/about_data</span>

 <span style="color:#888888;">Current Active OData (not used): data.cityofchicago.org/api/odata/v4/uupf-x98q?$format=json</span>

 <span style="color:#888888;">History Snapshot (not used in this notebook): data.cityofchicago.org/Community-Economic-Development/Business-License-History-Snapshot/r3b4-wv66/about_data</span>

 <span style="color:#888888;">History Snapshot OData (not used): data.cityofchicago.org/api/odata/v4/r3b4-wv66?$format=json</span>

2. Cook County Assessor - Assessed Values
- Portal: https://datacatalog.cookcountyil.gov/Property-Taxation/Assessor-Assessed-Values/uzyt-m557/about_data
- OData: https://datacatalog.cookcountyil.gov/api/odata/v4/uzyt-m557?$format=json

3. Local Parcel Shapefile (this project)
- Google shared Drive: https://drive.google.com/drive/folders/1JnzBO3kRZx88PXotE7Lj0sfB7yM7_v6Y?usp=drive_link
- Example: Parcels/Parcel_2016/Parcels.shp

## Official Data Links (with OData where available)


In [3]:
# Local-first data settings (default for this demo)
LOCAL_LICENSE_CSV = pick_first_existing([
    "Dataset/Business_Licenses_20260424.csv",
    "Business_Licenses_20260424.csv",
])

LOCAL_ASSESSED_CSV = pick_first_existing([
    "Dataset/Assessor_-_Assessed_Values_20260428.csv",
    "Assessor_-_Assessed_Values_20260428.csv",
])

print("Using local license file:", LOCAL_LICENSE_CSV)
print("Using local assessed file:", LOCAL_ASSESSED_CSV)

# Optional online references
LICENSE_ODATA = "https://data.cityofchicago.org/api/odata/v4/r5kz-chrr?$format=json"
LICENSE_RESOURCE = "https://data.cityofchicago.org/resource/r5kz-chrr.json"
ASSESSED_API_V3_JSON = "https://datacatalog.cookcountyil.gov/api/v3/views/uzyt-m557/query.json"
ASSESSED_ODATA = "https://datacatalog.cookcountyil.gov/api/odata/v4/uzyt-m557?$format=json"
ASSESSED_RESOURCE = "https://datacatalog.cookcountyil.gov/resource/uzyt-m557.json"

FileNotFoundError: None of these files exist: ['Dataset/Assessor_-_Assessed_Values_20260428.csv', 'Assessor_-_Assessed_Values_20260428.csv']

## 2-1. Business Licenses Dataset


In [ ]:
# Local-first preview (default path for class demo)
license_preview = pd.read_csv(LOCAL_LICENSE_CSV, nrows=20, low_memory=False)
print("Source: local CSV")

# Optional online preview (commented out intentionally)
# license_preview = pd.DataFrame(
#     soda_query(LICENSE_ODATA, {"$limit": "20"})
# )

print(f"Columns ({len(license_preview.columns)}):")
print(sorted(license_preview.columns))
print("\nFirst 20 rows:")
display(license_preview.head(20))

Source: local CSV
Columns (37):
['ACCOUNT NUMBER', 'ADDRESS', 'APPLICATION CREATED DATE', 'APPLICATION REQUIREMENTS COMPLETE', 'APPLICATION TYPE', 'BUSINESS ACTIVITY', 'BUSINESS ACTIVITY ID', 'CITY', 'COMMUNITY AREA', 'COMMUNITY AREA NAME', 'CONDITIONAL APPROVAL', 'DATE ISSUED', 'DOING BUSINESS AS NAME', 'ID', 'LATITUDE', 'LEGAL NAME', 'LICENSE APPROVED FOR ISSUANCE', 'LICENSE CODE', 'LICENSE DESCRIPTION', 'LICENSE ID', 'LICENSE NUMBER', 'LICENSE STATUS', 'LICENSE STATUS CHANGE DATE', 'LICENSE TERM EXPIRATION DATE', 'LICENSE TERM START DATE', 'LOCATION', 'LONGITUDE', 'NEIGHBORHOOD', 'PAYMENT DATE', 'POLICE DISTRICT', 'PRECINCT', 'SITE NUMBER', 'SSA', 'STATE', 'WARD', 'WARD PRECINCT', 'ZIP CODE']

First 20 rows:


,ID,LICENSE ID,ACCOUNT NUMBER,SITE NUMBER,LEGAL NAME,DOING BUSINESS AS NAME,ADDRESS,CITY,STATE,ZIP CODE,WARD,PRECINCT,WARD PRECINCT,POLICE DISTRICT,COMMUNITY AREA,COMMUNITY AREA NAME,NEIGHBORHOOD,LICENSE CODE,LICENSE DESCRIPTION,BUSINESS ACTIVITY ID,BUSINESS ACTIVITY,LICENSE NUMBER,APPLICATION TYPE,APPLICATION CREATED DATE,APPLICATION REQUIREMENTS COMPLETE,PAYMENT DATE,CONDITIONAL APPROVAL,LICENSE TERM START DATE,LICENSE TERM EXPIRATION DATE,LICENSE APPROVED FOR ISSUANCE,DATE ISSUED,LICENSE STATUS,LICENSE STATUS CHANGE DATE,SSA,LATITUDE,LONGITUDE,LOCATION
0,3082555-20260423,3082555,526348,1,JOE & SONS LAUNDRY LLC,COIN LAUNDRY AND CLEANERS,5885 S ARCHER AVE,CHICAGO,IL,60638,23.0,8.0,23-8,8.0,56.0,GARFIELD RIDGE,GARFIELD RIDGE,4404,Regulated Business License,692 | 771,Operation of a Laundromat - Self Service | Ret...,3082555,ISSUE,04/23/2026,04/23/2026,04/23/2026,N,04/23/2026,05/15/2028,04/23/2026,04/23/2026,AAI,NaN,NaN,41.795637,-87.762129,"(41.79563740522116, -87.76212930611456)"
1,3082489-20260423,3082489,526284,1,"CAL CLOSETS RETAIL, INC.",CALIFORNIA CLOSETS,730 N FRANKLIN ST 110,CHICAGO,IL,60654,42.0,15.0,42-15,18.0,8.0,NEAR NORTH SIDE,RIVER NORTH,1010,Limited Business License,602,Operation of an Administrative Commercial Office,3082489,ISSUE,04/21/2026,04/21/2026,04/22/2026,N,04/23/2026,05/15/2028,04/22/2026,04/23/2026,AAI,NaN,NaN,41.895623,-87.635825,"(41.89562283550215, -87.63582499692767)"
2,3082546-20260423,3082546,526344,1,FOREVER HONORING SERVICES LLC,FOREVER HONORING SERVICES LLC,5200 THATCHER RD,DOWNERS GROVE,IL,60515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1010,Limited Business License,963,Automatic Food Vending Machine Business,3082546,ISSUE,04/23/2026,04/23/2026,04/23/2026,N,04/23/2026,05/15/2028,04/23/2026,04/23/2026,AAI,NaN,NaN,NaN,NaN,NaN
3,2677836-20260516,3076449,425783,2,GAME NIGHT OUT LLC,GAME NIGHT OUT,2828 N CLARK ST 401,CHICAGO,IL,60657,44.0,18.0,44-18,19.0,6.0,LAKE VIEW,LAKE VIEW,1050,Public Place of Amusement,746 | 767,Charges a Fee for Entertainment or Amusements ...,2677836,RENEW,NaN,03/15/2026,03/24/2026,N,05/16/2026,05/15/2028,04/22/2026,04/23/2026,AAI,NaN,8.0,41.933474,-87.645599,"(41.93347403945103, -87.64559929166136)"
4,49452-20251216,3053017,37235,1,AMANAH ACCOUNTING SERVICE INC,AMANAH ACCOUNTING SERVICE INC,4042 W LAWRENCE AVE,CHICAGO,IL,60630,39.0,8.0,39-8,17.0,14.0,ALBANY PARK,ALBANY PARK,1010,Limited Business License,659,Provide Financial and Accounting Services,49452,RENEW,NaN,10/15/2025,12/11/2025,N,12/16/2025,12/15/2027,12/11/2025,04/23/2026,AAI,NaN,NaN,41.968332,-87.729793,"(41.96833187407959, -87.72979318160239)"
5,2595127-20260616,3081004,427030,1,"JAY JAY BHOLE, INC.",SUBWAY,1300 W CERMAK RD 1ST,CHICAGO,IL,60608,25.0,2.0,25-2,12.0,31.0,LOWER WEST SIDE,LOWER WEST SIDE,1006,Retail Food Establishment,781,Sale of Food Prepared Onsite With Dining Area,2595127,RENEW,NaN,04/15/2026,04/23/2026,N,06/16/2026,06/15/2028,04/23/2026,04/23/2026,AAI,NaN,NaN,41.852565,-87.658787,"(41.85256485189594, -87.65878714550385)"
6,2850727-20260616,3081512,486302,1,JONATHON'S SPORTSWEAR INC.,JONATHON'S,11121 S MICHIGAN AVE,CHICAGO,IL,60628,9.0,43.0,9-43,5.0,49.0,ROSELAND,ROSELAND,1010,Limited Business License,911,Retail Sales of Clothing / Accessories / Shoes,2850727,RENEW,NaN,04/15/2026,04/22/2026,N,06/16/2026,06/15/2028,04/22/2026,04/23/2026,AAI,NaN,40.0,41.691937,-87.620895,"(41.69193666237578, -87.62089498530223)"
7,2749984-20260501,3071285,371875,5,"CHICAGO DUFFY, LLC",CHICAGO DUFFY LLC,300 N STATE ST E E,CHICAGO,IL,60654,42.0,27.0,42-27,18.0,8.0,NEAR NORTH SIDE,RIVER NORTH,7020,Commercial Passenger Vessel,1012 | 1163,Operation of a Commercial Passenger Vessel - 2...,2749984,RENEW,NaN,02/15/2026,03/16/2026,N,05/01/2026,04/30/2027,04/23/2026,04/23/2026,AAI,NaN,NaN,41.887558,-87.628153,"(41.88755787281287, -87.62815306467208)"
8,2967350-20260516,3077227,503734,1,2358 CHICAGO INC.,2358 CHICAGO INC.,2358 W CHICAGO AVE 1ST,CHICAGO,IL,60622,36.0,12.0,36-12,12.0,24.0,WEST TOWN,UKRAINIAN VILLAGE,1010,Limited Business Lic


We first inspect business licenses from OData.

Suggested columns to pay attention to:
- `LICENSE ID`: unique license identifier
- `LEGAL NAME` / `DOING BUSINESS AS NAME`: entity name
- `LICENSE DESCRIPTION`: license type/category
- `LICENSE TERM START DATE` / `LICENSE TERM EXPIRATION DATE`: temporal validity
- `LICENSE STATUS`: status label
- `LATITUDE` / `LONGITUDE`: coordinates used for spatial matching
- `ADDRESS`: location string for QA checks

## 2-2. Parcel Dataset (Example Year: 2016)


In [ ]:

# Example of 2016 parcel shapefile reading with geopandas
ANALYSIS_YEAR = 2017
parcel_path = Path("Dataset/Parcels") / f"Parcel_{ANALYSIS_YEAR}" / "Parcels.shp"
parcels = gpd.read_file(parcel_path)

print(f"Parcel rows: {len(parcels):,}")
print(f"Parcel CRS: {parcels.crs}")
print("\nParcel preview (first 5 rows):")
display(parcels.head(5))

important_parcel_cols = [c for c in ["PIN10", "PARCELTYPE", "SHAPE_STAr", "SHAPE_STLe"] if c in parcels.columns]
print("\nSelected important columns:")
display(parcels[important_parcel_cols].head(10))

NameError: name 'Path' is not defined


In this demo, we use parcel polygons from `ANALYSIS_YEAR = 2016`.

Suggested columns to pay attention to:
- `PIN10`: parcel key used for cross-dataset mapping
- `PARCELTYPE`: parcel type code (not a direct commercial label)
- `SHAPE_STAr`: parcel area
- `SHAPE_STLe`: parcel perimeter
- `geometry`: polygon geometry used for spatial join

## 2-3. Why We Need PIN-Class Mapping

Problem:
- Licenses use coordinates (points)
- Parcels are polygons
- We also need a **commercial-only** parcel subset before matching

Key idea:
1. Use Assessed Values dataset to map `PIN -> class`
2. Select commercial classes
3. Join selected PINs to parcels (`PIN10`)
4. Then run spatial join: license points -> commercial parcel polygons

In [ ]:
# Fast assessed preview for class demo
# Use the official online endpoint first, because reading the full local CSV is slow.
# The full local file will only be loaded later in Step 1 when we actually need it.

import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen


def fetch_json_simple(url, params=None, timeout=40):
    params = params or {}
    sep = "&" if "?" in url else "?"
    full_url = url + (sep + urlencode(params) if params else "")
    req = Request(full_url, headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"})
    with urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


try:
    payload = fetch_json_simple(ASSESSED_API_V3_JSON, {"$limit": "20"})

    if isinstance(payload, list):
        assessed_preview = pd.DataFrame(payload)
    elif isinstance(payload, dict) and isinstance(payload.get("value"), list):
        assessed_preview = pd.DataFrame(payload["value"])
    elif isinstance(payload, dict) and isinstance(payload.get("data"), list):
        assessed_preview = pd.DataFrame(payload["data"])
    else:
        assessed_preview = pd.DataFrame()

    print("Source: assessed API v3 query.json")
except Exception as e:
    print(f"Online assessed preview failed ({e}). Fallback to local preview.")
    assessed_preview = pd.read_csv(LOCAL_ASSESSED_CSV, dtype=str, nrows=20, low_memory=False)
    print("Source: local CSV fallback")

print(f"Preview rows: {len(assessed_preview):,}")
print(f"Assessed values columns ({len(assessed_preview.columns)}):")
print(sorted([str(c) for c in assessed_preview.columns]))
print("\nFirst 20 rows:")
display(assessed_preview)

print("\nClass frequency preview is skipped here to keep this cell fast.")
print("The full assessed file will be loaded only in Step 1 when needed.")

Online assessed preview failed (HTTP Error 403: Forbidden). Fallback to local preview.
Source: local CSV fallback
Preview rows: 20
Assessed values columns (19):
['board_bldg', 'board_hie', 'board_land', 'board_tot', 'certified_bldg', 'certified_hie', 'certified_land', 'certified_tot', 'class', 'mailed_bldg', 'mailed_hie', 'mailed_land', 'mailed_tot', 'neighborhood_code', 'pin', 'row_id', 'tax_year', 'township_code', 'township_name']

First 20 rows:


,pin,tax_year,class,township_code,township_name,neighborhood_code,mailed_bldg,mailed_land,mailed_tot,mailed_hie,certified_bldg,certified_land,certified_tot,certified_hie,board_bldg,board_land,board_tot,board_hie,row_id
0,32213210090000,2020,202,12,Bloom,12111,"$2,287","$1,007","$3,294",$0,"$2,287","$1,007","$3,294",$0,"$2,287","$1,007","$3,294",$0,322132100900002020
1,32214030050000,2020,EX,12,Bloom,12111,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,322140300500002020
2,32214040370000,2020,EX,12,Bloom,12111,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,322140403700002020
3,32214110080000,2020,EX,12,Bloom,12111,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,322141100800002020
4,32214180030000,2020,EX,12,Bloom,12111,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,322141800300002020
5,32221040150000,2020,890,12,Bloom,12150,"$51,270","$15,652","$66,922",$0,"$21,407","$15,652","$37,059",$0,"$21,407","$15,652","$37,059",$0,322210401500002020
6,32231080510000,2020,EX,12,Bloom,12122,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,$0,322310805100002020
7,29052010120000,2016,203,37,Thornton,37024,"$3,962","$1,950","$5,912",NaN,"$3,962","$1,950","$5,912",NaN,"$3,962","$1,950","$5,912",NaN,290520101200002016
8,29054010360000,2016,202,37,Thornton,37024,"$4,585","$1,296","$5,881",NaN,"$4,585","$1,296","$5,881",NaN,"$4,585","$1,296","$5,881",NaN,290540103600002016
9,29054110460000,2016,203,37,Thornton,37024,"$5,578","$1,665","$7,243",NaN,"$5,578","$1,665","$7,243",NaN,"$5,578","$1,665","$7,243",NaN,290541104600002016



Class frequency preview is skipped here to keep this cell fast.
The full assessed file will be loaded only in Step 1 when needed.




Question: How do we align PIN formats between Assessor and Parcel data?

| Data Source | PIN Format | Example |
|---|---|---|
| Assessor CSV (`keypin`) | `XX-XX-XXX-XXX-XXXX` (with dashes, 18 chars) | `01-02-202-045-0000` |
| Parcel shapefile (`PIN10`) | Digits only, 10 chars | `0102202045` |

---

## 2-4. Commercial Class Filter

Below is the class strategy used in this tutorial (class-code only):

1. Primary whitelist (core commercial classes)
- Full 5-xx, 7-xx, 8-xx sets

2. Supplement whitelist
- Additional classes from the provided commercial valuation scope

**Primary whitelist details**
- A. Major Class 5A (Commercial)
- B. Major Class 7 (Commercial Incentive)
- C. Major Class 8 (Commercial/Industrial Incentive)

```text
5-00, 5-01, 5-16, 5-17, 5-22, 5-23, 5-26, 5-27, 5-28, 5-29,
5-30, 5-31, 5-32, 5-33, 5-35, 5-90, 5-91, 5-92, 5-97, 5-99,
7-00, 7-01, 7-16, 7-17, 7-22, 7-23, 7-26, 7-27, 7-28, 7-29,
7-30, 7-31, 7-32, 7-33, 7-35, 7-42, 7-43, 7-45, 7-46, 7-47,
7-48, 7-52, 7-53, 7-56, 7-57, 7-58, 7-60, 7-61, 7-62, 7-64,
7-65, 7-67, 7-72, 7-74, 7-90, 7-91, 7-92, 7-97, 7-98, 7-99,
8-00, 8-01, 8-16, 8-17, 8-22, 8-23, 8-27, 8-28, 8-29, 8-30,
8-31, 8-32, 8-33, 8-35, 8-90, 8-91, 8-92, 8-97, 8-99
```

**Supplement whitelist details**
- D. Mixed-use and partially commercial classes

```text
1-00, 1-90, 2-12, 2-25, 2-36, 3-00, 3-01, 3-13, 3-14, 3-15,
3-18, 3-19, 3-33, 3-40, 3-90, 3-91, 3-96, 3-97, 3-99, 4-17,
4-18, 4-27, 4-28, 4-47, 4-58, 4-90, 4-91, 4-92, 4-93, 4-97,
4-99, 6-00, 6-38, 6-54, 6-63, 6-70, 6-71, 6-73, 6-77, 6-79,
9-13, 9-14, 9-15, 9-18, 9-90, 9-91, 9-96, 9-97, 9-99
```

In [ ]:
# Commercial class lists (class-code only)
base_classes = [
    '5-00','5-01','5-16','5-17','5-22','5-23','5-26','5-27','5-28','5-29',
    '5-30','5-31','5-32','5-33','5-35','5-90','5-91','5-92','5-97','5-99',
    '7-00','7-01','7-16','7-17','7-22','7-23','7-26','7-27','7-28','7-29',
    '7-30','7-31','7-32','7-33','7-35','7-42','7-43','7-45','7-46','7-47',
    '7-48','7-52','7-53','7-56','7-57','7-58','7-60','7-61','7-62','7-64',
    '7-65','7-67','7-72','7-74','7-90','7-91','7-92','7-97','7-98','7-99',
    '8-00','8-01','8-16','8-17','8-22','8-23','8-27','8-28','8-29','8-30',
    '8-31','8-32','8-33','8-35','8-90','8-91','8-92','8-97','8-99',
]

extra_classes = [
    '1-00','1-90','2-12','2-25','2-36','3-00','3-01','3-13','3-14','3-15',
    '3-18','3-19','3-33','3-40','3-90','3-91','3-96','3-97','3-99','4-17',
    '4-18','4-27','4-28','4-47','4-58','4-90','4-91','4-92','4-93','4-97',
    '4-99','6-00','6-38','6-54','6-63','6-70','6-71','6-73','6-77','6-79',
    '9-13','9-14','9-15','9-18','9-90','9-91','9-96','9-97','9-99',
]

def class_hyphen_to_code(c: str) -> str:
    left, right = c.split('-')
    return f"{int(left)}{right}"

target_class_codes = sorted({class_hyphen_to_code(c) for c in (base_classes + extra_classes)})
print(f"Commercial class codes selected: {len(target_class_codes)}")

Commercial class codes selected: 128


---

# ▶️ Part 3. Combination Workflow (Step 1/2/3)

This section keeps the original workflow, but organizes it into smaller teaching steps so each cell has one clear purpose.

## Step 1 - Filting PINs by Commercial Classification

Dataset: Assessor - Assessed Values
https://datacatalog.cookcountyil.gov/api/odata/v4/uzyt-m557?$format=json

### **Step 1-A: build filtered pin-class records from local assessed CSV**

In [ ]:
r'''
import json
import time
import importlib
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

# -------------------------
# Step 1 - Assessed Values profiling + pin/class export
# Keeps the original output structure, while preserving both local and online options.
# -------------------------

STEP1_SOURCE_MODE = "local"  # change to "online" if you want API mode
OUT_DIR = Path("Outputs") / "step1_pins_class_mapping"
OUT_DIR.mkdir(parents=True, exist_ok=True)

STEP1_MAPPING_DIR = Path("Outputs") / "step1_pins_class_mapping"
STEP1_MAPPING_DIR.mkdir(parents=True, exist_ok=True)

pin_class_csv = OUT_DIR / "uzyt_m557_filtered_pin_class_unique.csv"
progress_json = OUT_DIR / "uzyt_m557_pin_class_export_progress.json"
class_stats_csv = OUT_DIR / "uzyt_m557_filtered_pin_class_class_stats.csv"
profile_csv = OUT_DIR / "uzyt_m557_api_profile_summary.csv"

selected_codes = set(target_class_codes)
class_where = "class in (" + ",".join([f"'{x}'" for x in target_class_codes]) + ")"

try:
    tqdm = importlib.import_module("tqdm.auto").tqdm
except Exception:
    tqdm = None


def normalize_class_code(v):
    s = "" if pd.isna(v) else str(v)
    return "".join(ch for ch in s if ch.isdigit())


def soda_query(params, retries=5, sleep_seconds=1.2):
    url = ASSESSED_RESOURCE + "?" + urlencode(params)
    headers = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}
    last_err = None
    for i in range(retries):
        try:
            with urlopen(Request(url, headers=headers), timeout=90) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except Exception as e:
            last_err = e
            if i < retries - 1:
                time.sleep(sleep_seconds * (i + 1))
            else:
                raise last_err


preview20 = pd.DataFrame()
preview50 = pd.DataFrame()
total_rows = 0
filtered_rows = 0
source_label = ""

if pin_class_csv.exists():
    pin_class_csv.unlink()

if STEP1_SOURCE_MODE == "local":
    assessed_header = pd.read_csv(LOCAL_ASSESSED_CSV, dtype=str, nrows=0)
    pin_col = next((c for c in assessed_header.columns if c.lower() in {"pin", "keypin"}), None)
    class_col = next((c for c in assessed_header.columns if c.lower() == "class"), None)
    if pin_col is None or class_col is None:
        raise KeyError(f"Required columns not found. pin_col={pin_col}, class_col={class_col}")

    preview20 = pd.read_csv(LOCAL_ASSESSED_CSV, dtype=str, nrows=20, low_memory=False)
    preview50 = pd.read_csv(LOCAL_ASSESSED_CSV, dtype=str, nrows=50, low_memory=False)
    source_label = f"local CSV: {LOCAL_ASSESSED_CSV}"

    chunk_reader = pd.read_csv(
        LOCAL_ASSESSED_CSV,
        dtype=str,
        usecols=[pin_col, class_col],
        chunksize=200000,
        low_memory=False,
    )
    chunk_iter = tqdm(chunk_reader, desc="Step 1 local assessed", unit="chunk") if tqdm is not None else chunk_reader
    write_header = True

    for chunk in chunk_iter:
        total_rows += len(chunk)
        chunk = chunk.rename(columns={pin_col: "pin", class_col: "class"}).copy()
        chunk["class_norm"] = chunk["class"].map(normalize_class_code)
        filtered_rows += int(chunk["class_norm"].isin(selected_codes).sum())
        chunk = chunk[chunk["class_norm"].isin(selected_codes)].copy()

        if chunk.empty:
            continue

        chunk["pin"] = chunk["pin"].astype(str).str.replace(r"\D", "", regex=True)
        chunk["class"] = chunk["class"].astype(str).str.strip()
        chunk["pin10_key"] = chunk["pin"].str[:10].str.zfill(10)
        export_chunk = chunk[["pin", "class", "pin10_key"]].dropna().drop_duplicates()
        export_chunk.to_csv(pin_class_csv, mode="a", header=write_header, index=False, encoding="utf-8-sig")
        write_header = False

    if pin_class_csv.exists():
        pin_class_df = pd.read_csv(pin_class_csv, dtype=str).drop_duplicates()
        pin_class_df.to_csv(pin_class_csv, index=False, encoding="utf-8-sig")
    else:
        pin_class_df = pd.DataFrame(columns=["pin", "class", "pin10_key"])

    progress_json.write_text(
        json.dumps(
            {
                "offset": int(total_rows),
                "exported": int(len(pin_class_df)),
                "class_where": class_where,
                "source_mode": STEP1_SOURCE_MODE,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
else:
    total_rows = int(soda_query({"$select": "count(*) as n"})[0]["n"])
    preview20 = pd.DataFrame(soda_query({"$limit": "20"}))
    preview50 = pd.DataFrame(soda_query({"$limit": "50"}))
    filtered_rows = int(soda_query({"$select": "count(*) as n", "$where": class_where})[0]["n"])
    source_label = f"online API: {ASSESSED_RESOURCE}"

    CHUNK_SIZE = 50000
    MAX_CHUNKS_THIS_RUN = 20

    if progress_json.exists() and pin_class_csv.exists():
        progress = json.loads(progress_json.read_text(encoding="utf-8"))
        offset = int(progress.get("offset", 0))
        exported = int(progress.get("exported", 0))
        write_header = False
    else:
        offset = 0
        exported = 0
        write_header = True

    chunks_done = 0
    while chunks_done < MAX_CHUNKS_THIS_RUN:
        rows = soda_query({
            "$select": "pin,class",
            "$where": class_where + " AND pin is not null AND class is not null",
            "$group": "pin,class",
            "$order": "pin,class",
            "$limit": str(CHUNK_SIZE),
            "$offset": str(offset),
        })
        if not rows:
            break

        df_chunk = pd.DataFrame(rows, columns=["pin", "class"]).dropna()
        if df_chunk.empty:
            break

        df_chunk["pin"] = df_chunk["pin"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(10)
        df_chunk["class"] = df_chunk["class"].astype(str).str.strip()
        df_chunk["pin10_key"] = df_chunk["pin"].str[:10].str.zfill(10)
        df_chunk = df_chunk[["pin", "class", "pin10_key"]].drop_duplicates()
        df_chunk.to_csv(pin_class_csv, mode="a", header=write_header, index=False, encoding="utf-8-sig")
        write_header = False

        got = len(df_chunk)
        exported += got
        offset += got
        chunks_done += 1

        progress_json.write_text(
            json.dumps(
                {
                    "offset": offset,
                    "exported": exported,
                    "class_where": class_where,
                    "source_mode": STEP1_SOURCE_MODE,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        print(f"Chunk {chunks_done}: +{got:,} rows | Exported unique pin-class rows: {exported:,}")

        if got < CHUNK_SIZE:
            break

    if pin_class_csv.exists():
        pin_class_df = pd.read_csv(pin_class_csv, dtype=str).drop_duplicates()
        pin_class_df.to_csv(pin_class_csv, index=False, encoding="utf-8-sig")
    else:
        pin_class_df = pd.DataFrame(columns=["pin", "class", "pin10_key"])

pin_class_stats = pin_class_df.groupby("class", dropna=False).size().reset_index(name="rows") if len(pin_class_df) else pd.DataFrame(columns=["class", "rows"])
if len(pin_class_stats):
    pin_class_stats["pct"] = (pin_class_stats["rows"] / max(len(pin_class_df), 1) * 100).round(4)
    pin_class_stats = pin_class_stats.sort_values("rows", ascending=False).reset_index(drop=True)
else:
    pin_class_stats["pct"] = pd.Series(dtype=float)

pin_class_stats.to_csv(class_stats_csv, index=False, encoding="utf-8-sig")

profile_df = pd.DataFrame([
    {
        "source_mode": STEP1_SOURCE_MODE,
        "source_label": source_label,
        "total_rows": int(total_rows),
        "filtered_rows_class_only": int(filtered_rows),
        "filtered_pct": round(filtered_rows / max(total_rows, 1) * 100, 6) if total_rows else 0,
        "exported_unique_pin_class_rows_current": int(len(pin_class_df)),
        "pin_class_csv": str(pin_class_csv),
        "pin_class_class_stats_csv": str(class_stats_csv),
        "progress_json": str(progress_json),
    }
])
profile_df.to_csv(profile_csv, index=False, encoding="utf-8-sig")

print("=== Step 1 Profile ===")
print(f"Source mode: {STEP1_SOURCE_MODE}")
print(f"Source: {source_label}")
print(f"Total rows: {total_rows:,}")
print(f"Filtered rows by class list: {filtered_rows:,}")
print(f"Exported unique pin-class rows: {len(pin_class_df):,}")
if tqdm is None and STEP1_SOURCE_MODE == "local":
    print("Progress bar not shown because tqdm is not installed.")

print("\n=== Preview: first 20 rows ===")
display(preview20)

print("\n=== Exported pin/class distribution (current) ===")
display(pin_class_stats.head(30))

print("\n=== Saved Outputs ===")
print(pin_class_csv)
print(class_stats_csv)
print(profile_csv)
'''

from pathlib import Path

import pandas as pd

STEP1_MAPPING_DIR = Path("Outputs") / "step1_pins_class_mapping"
step1_pin_class_csv = STEP1_MAPPING_DIR / "pin_class_filtered_unique.csv"

if not step1_pin_class_csv.exists():
    raise FileNotFoundError(
        f"Step 1-A export is disabled by default. Expected existing mapping file: {step1_pin_class_csv}"
    )

pin_class_df = pd.read_csv(step1_pin_class_csv, dtype=str).drop_duplicates()
print("Step 1-A export is disabled by default.")
print(f"Using existing mapping file: {step1_pin_class_csv}")
print(f"Rows loaded: {len(pin_class_df):,}")

### **Step 1-B: summarize and save**

In [ ]:
r'''
# Step 1B: summarize and mirror the exported files into the teaching output folder
pin_class_summary = (
    pin_class_df.groupby("pin10_key", dropna=False)
    .agg(
        class_count=("class", "nunique"),
        class_list=("class", lambda s: "|".join(sorted(pd.unique(s))))
    )
    .reset_index()
)

step1_pin_class_csv = STEP1_MAPPING_DIR / "pin_class_filtered_unique.csv"
step1_pin_summary_csv = STEP1_MAPPING_DIR / "pin_class_summary_by_pin10.csv"
out_pin_summary_csv = OUT_DIR / "uzyt_m557_pin_class_summary_by_pin10.csv"

pin_class_df.to_csv(step1_pin_class_csv, index=False, encoding="utf-8-sig")
pin_class_summary.to_csv(step1_pin_summary_csv, index=False, encoding="utf-8-sig")
pin_class_summary.to_csv(out_pin_summary_csv, index=False, encoding="utf-8-sig")

print("Saved Step 1 outputs:")
print(OUT_DIR / "uzyt_m557_filtered_pin_class_unique.csv")
print(class_stats_csv)
print(profile_csv)
print(out_pin_summary_csv)

print("\nSaved step1_pins_class_mapping mirror outputs:")
print(step1_pin_class_csv)
print(step1_pin_summary_csv)
print(f"Unique PIN10 keys: {pin_class_summary['pin10_key'].nunique():,}")
'''

from pathlib import Path

import pandas as pd

STEP1_MAPPING_DIR = Path("Outputs") / "step1_pins_class_mapping"
step1_pin_class_csv = STEP1_MAPPING_DIR / "pin_class_filtered_unique.csv"
if "pin_class_df" not in globals():
    if not step1_pin_class_csv.exists():
        raise FileNotFoundError(
            f"Step 1-B is disabled by default. Expected existing mapping file: {step1_pin_class_csv}"
        )
    pin_class_df = pd.read_csv(step1_pin_class_csv, dtype=str).drop_duplicates()

print("Step 1-B export is disabled by default.")
print(f"Only this Step 1 artifact is kept for downstream use: {step1_pin_class_csv}")
print("Other Step 1 derived files are no longer generated.")

## Step 2 - Parcel x Assessed Commercial Classes

### Step 2-A

This cell loads one parcel shapefile for the selected year and reads the filtered PIN-class table produced in Step 1.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

# -------------------------
# Step 2 - Parcel x filtered commercial PIN/class table
# -------------------------

step2_dir = Path("Outputs") / "step2_commercial_parcel"
step2_dir.mkdir(parents=True, exist_ok=True)

step1_source_candidates = [
    Path("Outputs") / "step1_pins_class_mapping" / "pin_class_filtered_unique.csv",
]

parcel_year = 2016  # 2018, 2020... choose a year you want to test
parcel_path_candidates = [
    Path("Parcels") / f"Parcel_{parcel_year}" / "Parcels.shp",
    Path("Dataset") / "Parcels" / f"Parcel_{parcel_year}" / "Parcels.shp",
]
parcel_path = next((p for p in parcel_path_candidates if p.exists()), None)
if parcel_path is None:
    raise FileNotFoundError(f"Parcel shapefile not found. Tried: {parcel_path_candidates}")

pin_class_path = next((p for p in step1_source_candidates if p.exists()), None)

parcels_year = gpd.read_file(parcel_path)
if pin_class_path is not None:
    pin_class_df = pd.read_csv(pin_class_path, dtype=str).drop_duplicates()
elif "pin_class_df" in globals():
    pin_class_df = pin_class_df.copy().drop_duplicates()
else:
    raise FileNotFoundError(f"Cannot find any Step 1 export. Tried: {step1_source_candidates}")

if "pin10_key" not in pin_class_df.columns:
    pin_class_df["pin10_key"] = (
        pin_class_df["pin"].astype(str).str.replace(r"\D", "", regex=True).str[:10].str.zfill(10)
    )

parcels_year["pin10_key"] = (
    parcels_year["PIN10"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(10)
)

print(f"Using parcel shapefile: {parcel_path}")
print(f"Using Step 1 export: {pin_class_path}")
print(f"Parcels ({parcel_year}) rows: {len(parcels_year):,}")
print(f"Unique PIN10 in filtered pin/class table: {pin_class_df['pin10_key'].nunique():,}")
print(f"CRS: {parcels_year.crs}")

### Step 2-B

This cell aggregates class information to the parcel level, keeps only commercial parcels, and saves the parcel outputs.

In [ ]:
# Step 2B: merge parcels with PIN-class summary and save outputs
pin_class_summary = (
    pin_class_df.dropna(subset=["pin10_key", "class"])
    .groupby("pin10_key")
    .agg(
        class_count=("class", "nunique"),
        class_list=("class", lambda s: "|".join(sorted(pd.unique(s))))
    )
    .reset_index()
)

commercial_parcels_year = parcels_year.merge(pin_class_summary, on="pin10_key", how="inner")
if parcel_year == 2016:
    commercial_parcels_2016 = commercial_parcels_year.copy()

class_count_dist = (
    commercial_parcels_year["class_count"]
    .value_counts()
    .sort_index()
    .rename_axis("class_count")
    .reset_index(name="parcel_rows")
)
class_count_dist["pct"] = (class_count_dist["parcel_rows"] / max(len(commercial_parcels_year), 1) * 100).round(4)

print("=== Step 2 Stats: Parcel x Commercial PINs ===")
print(f"Parcel {parcel_year} total rows: {len(parcels_year):,}")
print(f"Matched commercial parcels ({parcel_year}): {len(commercial_parcels_year):,}")
print(f"Parcel match coverage: {len(commercial_parcels_year) / max(len(parcels_year), 1):.2%}")

print("\n=== How many classes per parcel ===")
display(class_count_dist)

display_cols = [c for c in ["PIN10", "pin10_key", "class_count", "class_list", "PARCELTYPE", "SHAPE_STAr"] if c in commercial_parcels_year.columns]
print("\n=== Preview of matched commercial parcels ===")
display(commercial_parcels_year[display_cols].head(20))

step2_gpkg = step2_dir / f"parcels_{parcel_year}_commercial.gpkg"
# step2_csv = step2_dir / f"parcels_{parcel_year}_commercial_attributes.csv"
# step2_class_count_csv = step2_dir / f"parcels_{parcel_year}_commercial_classcount_distribution.csv"

commercial_parcels_year.to_file(step2_gpkg, driver="GPKG")
# commercial_parcels_year.drop(columns="geometry", errors="ignore").to_csv(step2_csv, index=False, encoding="utf-8-sig")
# class_count_dist.to_csv(step2_class_count_csv, index=False, encoding="utf-8-sig")

print("\nSaved Step 2 outputs:")
print(step2_gpkg)
print("Step 2 flat-file CSV exports are disabled because downstream cells only use the GeoPackage.")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Optional preview map for commercial parcels
commercial_gpkg = step2_dir / f"parcels_{parcel_year}_commercial.gpkg"

if commercial_gpkg.exists():
    commercial_parcels_plot = gpd.read_file(commercial_gpkg)
    print(f"Loaded existing data: {commercial_gpkg}")
elif "commercial_parcels_year" in globals():
    commercial_parcels_plot = commercial_parcels_year.copy()
    print("Using in-memory variable: commercial_parcels_year")
else:
    raise FileNotFoundError(f"Cannot find {commercial_gpkg}. Run Step 2 once to generate it.")

preview_n = min(5000, len(commercial_parcels_plot))
preview_plot = (
    commercial_parcels_plot.sample(preview_n, random_state=42)
    if len(commercial_parcels_plot) > preview_n
    else commercial_parcels_plot
)

fig, ax = plt.subplots(figsize=(10, 10), facecolor="#000000")
ax.set_facecolor("#000000")
preview_plot.boundary.plot(ax=ax, linewidth=0.10, color="#1f2937", alpha=0.20, zorder=1)
preview_plot.plot(ax=ax, linewidth=0.30, edgecolor="#ffffff", facecolor="#a5f3fc", alpha=0.95, zorder=2)
centroids = preview_plot.geometry.centroid
ax.scatter(centroids.x, centroids.y, s=3.5, c="#ecfeff", alpha=0.75, linewidths=0, zorder=3)
ax.set_title(f"{parcel_year} Commercial Parcels Preview (sample={preview_n:,})", color="#ffffff", pad=12)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Step 3. Match Business Licenses to Commercial Parcels

In the next cells, we will first prepare the license table for one year, then run the spatial join, and finally save the matched outputs.

### Step 3-A business license data reading

This cell reads business license data, standardizes the important columns, and filters the records to the selected analysis year.

In [ ]:
import json
import time
from urllib.parse import urlencode
from urllib.request import Request, urlopen

# -------------------------
# Step 3A - load license records for one year
# Preserves both local and online options.
# -------------------------

STEP3_LICENSE_SOURCE_MODE = "local"  # change to "online" if you want portal API mode
analysis_year = 2016
analysis_start = pd.Timestamp(f"{analysis_year}-01-01")
analysis_end = pd.Timestamp(f"{analysis_year}-12-31")
today = pd.Timestamp.today().normalize()


def license_query(params, retries=5, sleep_seconds=1.2):
    url = LICENSE_RESOURCE + "?" + urlencode(params)
    headers = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}
    last_err = None
    for i in range(retries):
        try:
            with urlopen(Request(url, headers=headers), timeout=90) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except Exception as e:
            last_err = e
            if i < retries - 1:
                time.sleep(sleep_seconds * (i + 1))
            else:
                raise last_err


def standardize_license_columns(df):
    expected = {
        "license id": "LICENSE ID",
        "legal name": "LEGAL NAME",
        "doing business as name": "DOING BUSINESS AS NAME",
        "license description": "LICENSE DESCRIPTION",
        "address": "ADDRESS",
        "license term start date": "LICENSE TERM START DATE",
        "license term expiration date": "LICENSE TERM EXPIRATION DATE",
        "date issued": "DATE ISSUED",
        "license status": "LICENSE STATUS",
        "latitude": "LATITUDE",
        "longitude": "LONGITUDE",
    }
    rename_map = {}
    for col in df.columns:
        key = str(col).strip().lower().replace("_", " ")
        if key in expected:
            rename_map[col] = expected[key]
    df = df.rename(columns=rename_map)
    for target in expected.values():
        if target not in df.columns:
            df[target] = pd.NA
    return df


def load_license_source(mode="local", limit_per_page=50000, max_pages=None):
    if mode == "local":
        df = pd.read_csv(
            LOCAL_LICENSE_CSV,
            dtype=str,
            low_memory=False,
            keep_default_na=False,
            na_values=["", "NA", "N/A", "NULL"],
        )
        return standardize_license_columns(df), f"local CSV: {LOCAL_LICENSE_CSV}"

    rows = []
    offset = 0
    pages_done = 0
    while True:
        batch = license_query({"$limit": str(limit_per_page), "$offset": str(offset)})
        if not batch:
            break
        rows.extend(batch)
        got = len(batch)
        offset += got
        pages_done += 1
        print(f"Fetched online license rows: {offset:,}")
        if got < limit_per_page:
            break
        if max_pages is not None and pages_done >= max_pages:
            break

    return standardize_license_columns(pd.DataFrame(rows)), f"online API: {LICENSE_RESOURCE}"


licenses, license_source_label = load_license_source(STEP3_LICENSE_SOURCE_MODE)
for col in ["LICENSE TERM START DATE", "LICENSE TERM EXPIRATION DATE", "DATE ISSUED"]:
    licenses[col] = pd.to_datetime(licenses[col], errors="coerce")
for col in ["LATITUDE", "LONGITUDE"]:
    licenses[col] = pd.to_numeric(licenses[col], errors="coerce")

licenses["start_eff"] = licenses["LICENSE TERM START DATE"].fillna(licenses["DATE ISSUED"])
licenses["is_valid_in_2016"] = (
    licenses["start_eff"].notna()
    & (licenses["start_eff"] <= analysis_end)
    & (licenses["LICENSE TERM EXPIRATION DATE"].isna() | (licenses["LICENSE TERM EXPIRATION DATE"] >= analysis_start))
)
licenses["active_now"] = (
    licenses["LICENSE TERM EXPIRATION DATE"].isna()
    | (licenses["LICENSE TERM EXPIRATION DATE"] >= today)
)
licenses["license_group"] = licenses["active_now"].map({True: "active_now", False: "historic_now"})

licenses_2016 = licenses.loc[licenses["is_valid_in_2016"]].copy()
licenses_2016_geo = licenses_2016.loc[licenses_2016["LATITUDE"].notna() & licenses_2016["LONGITUDE"].notna()].copy()

print("=== Step 3 Source ===")
print(license_source_label)
print(f"License sample rows: {len(licenses):,}")
print(f"Valid in {analysis_year}: {len(licenses_2016):,}")
print(f"Valid + geo rows: {len(licenses_2016_geo):,}")

### Step 3-B: spatial join

In [ ]:
# Step 3B: spatial join and inspect the matched result
commercial_parcel_candidates = [
    step2_dir / f"parcels_{analysis_year}_commercial.gpkg",
    Path("Outputs") / "step2_commercial_parcel" / f"parcels_{analysis_year}_commercial.gpkg",
]
commercial_parcel_gpkg = next((p for p in commercial_parcel_candidates if p.exists()), commercial_parcel_candidates[0])
if "commercial_parcels_year" in globals() and len(commercial_parcels_year):
    commercial_parcels_for_join = commercial_parcels_year.copy()
else:
    commercial_parcels_for_join = gpd.read_file(commercial_parcel_gpkg)

parcel_keep_cols = [
    col for col in ["PIN10", "pin10_key", "class_count", "class_list", "PARCELTYPE", "SHAPE_STAr", "geometry"]
    if col in commercial_parcels_for_join.columns
]
commercial_parcels_for_join = commercial_parcels_for_join[parcel_keep_cols].copy()

licenses_2016_points = gpd.GeoDataFrame(
    licenses_2016_geo,
    geometry=gpd.points_from_xy(licenses_2016_geo["LONGITUDE"], licenses_2016_geo["LATITUDE"]),
    crs="EPSG:4326",
).to_crs(commercial_parcels_for_join.crs)

licenses_2016_matched = gpd.sjoin(
    licenses_2016_points,
    commercial_parcels_for_join,
    how="inner",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")

summary_rows = [
    {"metric": "total_full_license_rows", "value": int(len(licenses))},
    {"metric": "valid_in_2016_rows", "value": int(len(licenses_2016))},
    {"metric": "valid_in_2016_with_latlon_rows", "value": int(len(licenses_2016_geo))},
    {"metric": "matched_to_commercial_parcel_rows", "value": int(len(licenses_2016_matched))},
    {"metric": "match_rate_over_valid_with_latlon", "value": round(len(licenses_2016_matched) / max(len(licenses_2016_geo), 1), 6)},
    {"metric": "unique_matched_license_ids", "value": int(licenses_2016_matched["LICENSE ID"].nunique()) if "LICENSE ID" in licenses_2016_matched.columns else int(len(licenses_2016_matched))},
    {"metric": "unique_matched_pin10", "value": int(licenses_2016_matched["PIN10"].nunique()) if "PIN10" in licenses_2016_matched.columns else -1},
]
summary_df = pd.DataFrame(summary_rows)

license_group_stats = (
    licenses_2016_matched["license_group"]
    .value_counts(dropna=False)
    .rename_axis("license_group")
    .reset_index(name="rows")
)
license_group_stats["pct"] = (license_group_stats["rows"] / max(len(licenses_2016_matched), 1) * 100).round(4)

print("=== Step 3 Stats: Business Licenses x Commercial Parcels ===")
for row in summary_rows:
    metric = row["metric"]
    value = row["value"]
    if isinstance(value, float) and value <= 1:
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value:,}" if isinstance(value, int) else f"{metric}: {value}")

print("\n=== Matched license group distribution ===")
display(license_group_stats)

preview_cols = [
    col for col in [
        "LICENSE ID", "LEGAL NAME", "DOING BUSINESS AS NAME", "LICENSE DESCRIPTION",
        "ADDRESS", "LICENSE TERM START DATE", "LICENSE TERM EXPIRATION DATE",
        "LICENSE STATUS", "license_group", "PIN10", "class_count", "class_list", "PARCELTYPE"
    ]
    if col in licenses_2016_matched.columns
]
print("\n=== Preview of matched licenses ===")
display(licenses_2016_matched[preview_cols].head(20))

licenses_2016_plain = pd.DataFrame(licenses_2016)
licenses_2016_matched_plain = pd.DataFrame(licenses_2016_matched.drop(columns="geometry", errors="ignore"))


### Step 3-C: save files and map preview


In [ ]:
import matplotlib.pyplot as plt

# Step 3C: save files and optionally draw a quick map preview
step3_dir = Path("Outputs") / "step3_license_parquet"
step3_dir.mkdir(parents=True, exist_ok=True)

matched_geo_parquet = step3_dir / f"licenses_{analysis_year}_matched_commercial_geo.parquet"
# all_parquet = step3_dir / f"licenses_{analysis_year}_all.parquet"
# all_csv = step3_dir / f"licenses_{analysis_year}_all.csv"
# matched_parquet = step3_dir / f"licenses_{analysis_year}_matched_commercial.parquet"
# matched_csv = step3_dir / f"licenses_{analysis_year}_matched_commercial.csv"
# stats_csv = step3_dir / f"licenses_{analysis_year}_match_summary.csv"

licenses_2016_matched.to_parquet(matched_geo_parquet, index=False)
# licenses_2016_plain.to_parquet(all_parquet, index=False)
# licenses_2016_plain.to_csv(all_csv, index=False, encoding="utf-8-sig")
# licenses_2016_matched_plain.to_parquet(matched_parquet, index=False)
# licenses_2016_matched_plain.to_csv(matched_csv, index=False, encoding="utf-8-sig")
# summary_df.to_csv(stats_csv, index=False, encoding="utf-8-sig")

print("\nSaved Step 3 outputs:")
print(matched_geo_parquet)
print("Other Step 3 flat-file exports are currently commented out.")

if len(licenses_2016_matched) > 0:
    point_preview_n = min(20000, len(licenses_2016_matched))
    points_preview = licenses_2016_matched.sample(point_preview_n, random_state=42) if len(licenses_2016_matched) > point_preview_n else licenses_2016_matched

    fig, ax = plt.subplots(figsize=(10, 10), facecolor="#020617")
    ax.set_facecolor("#020617")
    commercial_parcels_for_join.boundary.plot(ax=ax, linewidth=0.5, color="#ADCFFF", alpha=0.7, zorder=1)
    points_preview.plot(ax=ax, markersize=3.2, color="#fff3b7", alpha=0.9, zorder=2)
    ax.set_title(f"{analysis_year} Matched License Addresses (sample={point_preview_n:,})", color="#f8fafc", pad=12)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
else:
    print("No matched license addresses to plot.")


# ▶️ Part 4. Multi-Year Extension

This part keeps the same output structure from Part 3, but applies the workflow across all available parcel years.

## 4-1. Detect Available Years and Output Paths

This cell finds which parcel years are available in the workspace and prepares the folders used for multi-year exports.

**Part 4 Run Guide**

If you only want to run the multi-year pipeline, use this order:

| Stage | Required | Cell | Title | Purpose |
|---|---|---:|---|---|
| Setup | Yes | 11 | Local data settings | Define local CSV paths and online endpoint constants |
| Setup | Yes | 22 | Commercial class-code list | Build the commercial class whitelist used by the pipeline |
| Setup | Yes | 26 | Step 1-A | Load `Outputs/step1_pins_class_mapping/pin_class_filtered_unique.csv` |
| Setup | Yes | 37 | Step 3-A business license data reading | Define `load_license_source()` for license loading |
| Part 4 | Yes | 46 | Method 1. Local setup and output paths | Initialize local parcel root, detect local years, and create output folders |
| Part 4 | Optional | 48 | Method 2. Google Drive inspection | Add online parcel-year metadata and inspect Google Drive archive contents |
| Part 4 | Yes | 50 | 4-2. Define the Reusable One-Year Function | Build the reusable multi-year pipeline function |
| Part 4 | Yes | 52 | 4-3. Run the Workflow for All Years | Actually run the multi-year pipeline |



### Method 1. Local setup and output paths

In [ ]:
parcel_root_candidates = [Path("Parcels"), Path("Dataset") / "Parcels"]
parcel_root = next((p for p in parcel_root_candidates if p.exists()), None)
if parcel_root is None:
    raise FileNotFoundError(f"Parcel root not found. Tried: {parcel_root_candidates}")

all_detected_years = sorted(
    int(p.name.split("_")[-1])
    for p in parcel_root.glob("Parcel_*")
    if p.is_dir() and p.name.split("_")[-1].isdigit()
)

TARGET_YEARS = all_detected_years.copy()  # you can override manually, e.g. [2016, 2017]
PIPELINE_STEP2_DIR = Path("Outputs") / "step2_commercial_parcel"
PIPELINE_STEP3_DIR = Path("Outputs") / "step3_license_parquet"
PIPELINE_STEP2_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_STEP3_DIR.mkdir(parents=True, exist_ok=True)
PROCESS_ALL_LOG = PIPELINE_STEP3_DIR / "process_all_years_log.csv"

print("Using parcel root:", parcel_root)
print("Detected years:", TARGET_YEARS)
print("Step 2 output root:", PIPELINE_STEP2_DIR)
print("Step 3 output root:", PIPELINE_STEP3_DIR)


### Method 2. Optional Google Drive inspection

Inspect the shared Google Drive folder and the inside of each `Parcel_YYYY.zip` without downloading the full archive. This is useful for checking whether a year contains a shapefile package (`.shp/.dbf/.shx/.prj`) or only tables such as `.csv` / `.geojson`.

In [ ]:
import re
import struct
from html import unescape

import pandas as pd
import requests

PARCEL_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1JnzBO3kRZx88PXotE7Lj0sfB7yM7_v6Y"


def list_drive_parcel_archives(folder_url: str = PARCEL_DRIVE_FOLDER_URL) -> pd.DataFrame:
    html = requests.get(folder_url, timeout=30).text
    pattern = re.compile(
        r'\[null,&quot;(?P<file_id>[A-Za-z0-9_-]{20,})&quot;\],null,null,null,&quot;application/zip&quot;.*?\[\[\[&quot;(?P<name>Parcel_\d{4}\.zip)&quot;',
        re.S,
    )
    rows = [
        {"name": name, "file_id": file_id, "year": int(name[7:11])}
        for file_id, name in pattern.findall(html)
    ]
    return (
        pd.DataFrame(rows)
        .drop_duplicates(subset=["name", "file_id"])
        .sort_values("year")
        .reset_index(drop=True)
    )


def resolve_drive_download(file_id: str) -> tuple[str, dict[str, str]]:
    page = requests.get(f"https://drive.google.com/uc?export=download&id={file_id}", timeout=30)
    if "Virus scan warning" not in page.text:
        return page.url, {}

    action = re.search(r'<form[^>]+id="download-form"[^>]+action="([^"]+)"', page.text)
    if action is None:
        raise ValueError(f"Could not parse Google Drive download form for file id={file_id}")

    params = {
        match.group(1): unescape(match.group(2))
        for match in re.finditer(
            r'<input[^>]+type="hidden"[^>]+name="([^"]+)"[^>]+value="([^"]*)"',
            page.text,
        )
    }
    return action.group(1), params


def inspect_drive_zip_members(file_id: str, tail_bytes: int = 262144) -> list[str]:
    download_url, params = resolve_drive_download(file_id)
    response = requests.get(
        download_url,
        params=params,
        headers={"Range": f"bytes=-{tail_bytes}"},
        timeout=30,
    )
    response.raise_for_status()
    content_range = response.headers.get("Content-Range")
    if content_range is None:
        raise ValueError("Google Drive did not return a partial-content response for zip inspection.")

    data = response.content
    eocd_pos = data.rfind(b"PK\x05\x06")
    if eocd_pos == -1:
        raise ValueError("Could not locate the zip central directory in the downloaded tail bytes.")

    _, _, _, _, _, cd_size, cd_offset, _ = struct.unpack("<4s4H2LH", data[eocd_pos : eocd_pos + 22])
    total_size = int(content_range.split("/")[-1])
    start_offset = total_size - len(data)
    cd_start = cd_offset - start_offset
    central_directory = data[cd_start : cd_start + cd_size]

    members = []
    cursor = 0
    while cursor < len(central_directory) and central_directory[cursor : cursor + 4] == b"PK\x01\x02":
        header = struct.unpack("<4s6H3L5H2L", central_directory[cursor : cursor + 46])
        name_len, extra_len, comment_len = header[10], header[11], header[12]
        member = central_directory[cursor + 46 : cursor + 46 + name_len].decode("utf-8", errors="replace")
        members.append(member)
        cursor += 46 + name_len + extra_len + comment_len

    return members


def summarize_drive_zip_members(archive_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for archive in archive_df.itertuples(index=False):
        members = inspect_drive_zip_members(archive.file_id)
        extensions = sorted({member.rsplit(".", 1)[-1].lower() for member in members if "." in member})
        rows.append(
            {
                "name": archive.name,
                "extensions": ", ".join(extensions),
            }
        )
    return pd.DataFrame(rows)


parcel_drive_archives = list_drive_parcel_archives()
drive_zip_summary = summarize_drive_zip_members(parcel_drive_archives)
if "TARGET_YEARS" in globals():
    TARGET_YEARS = sorted(set(TARGET_YEARS) | set(parcel_drive_archives["year"].tolist()))
    print("Updated TARGET_YEARS with Google Drive archives:", TARGET_YEARS)
print("Google Drive parcel archive preview:")
display(drive_zip_summary)

## 4-2. Define the Reusable One-Year Function

This cell wraps the same Step 2 and Step 3 logic into one function so the workflow can be repeated for each parcel year.

In [ ]:
import importlib
import io
import zipfile
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests


PIPELINE_CACHE_DIR = Path("Dataset") / "Parcels"
PIPELINE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

try:
    pipeline_tqdm = importlib.import_module("tqdm.auto").tqdm
except Exception:
    pipeline_tqdm = None


DRIVE_VECTOR_PREFERENCE = [".shp", ".geojson", ".json"]


def get_drive_archive_row(year: int):
    if "parcel_drive_archives" not in globals():
        raise RuntimeError("Run Cell 45 first to load Google Drive parcel archive metadata.")
    matched = parcel_drive_archives.loc[parcel_drive_archives["year"] == year]
    return matched.iloc[0] if len(matched) else None


def choose_drive_vector_member(file_id: str) -> str | None:
    members = inspect_drive_zip_members(file_id)
    for suffix in DRIVE_VECTOR_PREFERENCE:
        for member in members:
            if member.lower().endswith(suffix):
                return member
    return None


def download_drive_archive(file_id: str, archive_name: str) -> Path:
    archive_dir = PIPELINE_CACHE_DIR / archive_name.replace(".zip", "")
    archive_dir.mkdir(parents=True, exist_ok=True)
    archive_zip_path = archive_dir / archive_name
    if archive_zip_path.exists() and archive_zip_path.stat().st_size > 0:
        return archive_zip_path

    download_url, params = resolve_drive_download(file_id)
    with requests.get(download_url, params=params, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(archive_zip_path, "wb") as output_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output_file.write(chunk)
    return archive_zip_path


def load_parcels_from_drive(year: int) -> tuple[gpd.GeoDataFrame, str, str]:
    archive = get_drive_archive_row(year)
    if archive is None:
        raise FileNotFoundError(f"Google Drive archive metadata not found for year={year}")

    archive_name = str(archive["name"])
    file_id = str(archive["file_id"])
    member_name = choose_drive_vector_member(file_id)
    if member_name is None:
        raise FileNotFoundError(f"No supported parcel geometry file found in Google Drive archive for year={year}")

    archive_zip_path = download_drive_archive(file_id, archive_name)
    if member_name.lower().endswith(".shp"):
        parcels = gpd.read_file(f"zip://{archive_zip_path}!{member_name}")
        source_label = f"google_drive_zip_shp:{archive_name}!{member_name}"
    else:
        with zipfile.ZipFile(archive_zip_path) as zip_file:
            with zip_file.open(member_name) as member_file:
                parcels = gpd.read_file(io.BytesIO(member_file.read()))
        source_label = f"google_drive_zip_geojson:{archive_name}!{member_name}"

    return parcels, source_label, member_name


def load_parcels_local_fallback(year: int) -> tuple[gpd.GeoDataFrame, str, str]:
    parcel_path_candidates = [
        parcel_root / f"Parcel_{year}" / "Parcels.shp",
        Path("Parcels") / f"Parcel_{year}" / "Parcels.shp",
        Path("Dataset") / "Parcels" / f"Parcel_{year}" / "Parcels.shp",
    ]
    parcel_path = next((p for p in parcel_path_candidates if p.exists()), None)
    if parcel_path is None:
        raise FileNotFoundError(f"Local parcel shapefile not found for year={year}. Tried: {parcel_path_candidates}")
    return gpd.read_file(parcel_path), f"local_shapefile:{parcel_path}", str(parcel_path.name)


def load_parcels_for_pipeline(year: int, prefer_drive: bool = True) -> tuple[gpd.GeoDataFrame, str, str]:
    errors = []
    if prefer_drive:
        try:
            return load_parcels_from_drive(year)
        except Exception as exc:
            errors.append(f"drive={exc}")
    try:
        return load_parcels_local_fallback(year)
    except Exception as exc:
        errors.append(f"local={exc}")
    raise FileNotFoundError(f"No parcel source available for year={year}. {' | '.join(errors)}")


def run_one_year_pipeline(year: int, licenses_full: pd.DataFrame, pin_class_source_df: pd.DataFrame, skip_existing: bool = True):
    commercial_gpkg = PIPELINE_STEP2_DIR / f"parcels_{year}_commercial.gpkg"
    matched_geo_parquet = PIPELINE_STEP3_DIR / f"licenses_{year}_matched_commercial_geo.parquet"

    if skip_existing and commercial_gpkg.exists() and matched_geo_parquet.exists():
        return {
            "year": year,
            "status": "skipped_existing",
            "parcel_source": "existing_outputs",
            "parcel_member": "",
            "parcel_rows": pd.NA,
            "commercial_parcel_rows": pd.NA,
            "license_geo_rows": pd.NA,
            "matched_rows": pd.NA,
            "match_rate": pd.NA,
            "commercial_gpkg": str(commercial_gpkg),
            "matched_geo_parquet": str(matched_geo_parquet),
        }

    try:
        parcels_y, parcel_source_label, parcel_member = load_parcels_for_pipeline(year, prefer_drive=True)
    except FileNotFoundError as exc:
        return {
            "year": year,
            "status": "skipped_no_parcel_source",
            "parcel_source": "",
            "parcel_member": "",
            "parcel_rows": pd.NA,
            "commercial_parcel_rows": pd.NA,
            "license_geo_rows": pd.NA,
            "matched_rows": pd.NA,
            "match_rate": pd.NA,
            "commercial_gpkg": str(commercial_gpkg),
            "matched_geo_parquet": str(matched_geo_parquet),
            "error": str(exc),
        }

    if parcels_y.empty:
        return {
            "year": year,
            "status": "skipped_empty_parcel_geometry",
            "parcel_source": parcel_source_label,
            "parcel_member": parcel_member,
            "parcel_rows": 0,
            "commercial_parcel_rows": pd.NA,
            "license_geo_rows": pd.NA,
            "matched_rows": pd.NA,
            "match_rate": pd.NA,
            "commercial_gpkg": str(commercial_gpkg),
            "matched_geo_parquet": str(matched_geo_parquet),
        }

    pin_col = next((col for col in parcels_y.columns if str(col).strip().upper() == "PIN10"), None)
    if pin_col is None:
        raise KeyError(f"Parcel source for year={year} does not contain a PIN10-like column")
    parcels_y["pin10_key"] = parcels_y[pin_col].astype(str).str.replace(r"\D", "", regex=True).str.zfill(10)

    pin_class_work = pin_class_source_df.copy()
    if "pin10_key" not in pin_class_work.columns:
        pin_class_work["pin10_key"] = (
            pin_class_work["pin"].astype(str).str.replace(r"\D", "", regex=True).str[:10].str.zfill(10)
        )

    pin_class_summary_y = (
        pin_class_work.dropna(subset=["pin10_key", "class"])
        .groupby("pin10_key")
        .agg(
            class_count=("class", "nunique"),
            class_list=("class", lambda s: "|".join(sorted(pd.unique(s))))
        )
        .reset_index()
    )

    commercial_y = parcels_y.merge(pin_class_summary_y, on="pin10_key", how="inner")
    commercial_y.to_file(commercial_gpkg, driver="GPKG")

    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year}-12-31")
    today_local = pd.Timestamp.today().normalize()

    lic_y = licenses_full.copy()
    lic_y["start_eff"] = lic_y["LICENSE TERM START DATE"].fillna(lic_y["DATE ISSUED"])
    lic_y["is_valid_in_year"] = (
        lic_y["start_eff"].notna()
        & (lic_y["start_eff"] <= end)
        & (lic_y["LICENSE TERM EXPIRATION DATE"].isna() | (lic_y["LICENSE TERM EXPIRATION DATE"] >= start))
    )
    lic_y["active_now"] = (
        lic_y["LICENSE TERM EXPIRATION DATE"].isna()
        | (lic_y["LICENSE TERM EXPIRATION DATE"] >= today_local)
    )
    lic_y["license_group"] = lic_y["active_now"].map({True: "active_now", False: "historic_now"})

    licenses_year = lic_y.loc[lic_y["is_valid_in_year"]].copy()
    licenses_year_geo = licenses_year.loc[licenses_year["LATITUDE"].notna() & licenses_year["LONGITUDE"].notna()].copy()

    licenses_points = gpd.GeoDataFrame(
        licenses_year_geo,
        geometry=gpd.points_from_xy(licenses_year_geo["LONGITUDE"], licenses_year_geo["LATITUDE"]),
        crs="EPSG:4326",
    ).to_crs(commercial_y.crs)

    licenses_matched = gpd.sjoin(
        licenses_points,
        commercial_y,
        how="inner",
        predicate="within",
    ).drop(columns=["index_right"], errors="ignore")

    licenses_matched.to_parquet(matched_geo_parquet, index=False)

    return {
        "year": year,
        "status": "ok",
        "parcel_source": parcel_source_label,
        "parcel_member": parcel_member,
        "parcel_rows": int(len(parcels_y)),
        "commercial_parcel_rows": int(len(commercial_y)),
        "license_geo_rows": int(len(licenses_year_geo)),
        "matched_rows": int(len(licenses_matched)),
        "match_rate": round(len(licenses_matched) / max(len(licenses_year_geo), 1), 6),
        "commercial_gpkg": str(commercial_gpkg),
        "matched_geo_parquet": str(matched_geo_parquet),
    }

## 4-3. Run the Workflow for All Years

This final cell reuses the one-year function, writes the yearly files, and saves a process log for quick checking.

In [ ]:
PIPELINE_LICENSE_SOURCE_MODE = "local"  # change to "online" if needed
PIPELINE_SKIP_EXISTING = True

licenses_full, pipeline_license_source_label = load_license_source(PIPELINE_LICENSE_SOURCE_MODE)
for col in ["LICENSE TERM START DATE", "LICENSE TERM EXPIRATION DATE", "DATE ISSUED"]:
    licenses_full[col] = pd.to_datetime(licenses_full[col], errors="coerce")
for col in ["LATITUDE", "LONGITUDE"]:
    licenses_full[col] = pd.to_numeric(licenses_full[col], errors="coerce")

pin_class_source_candidates = [
    Path("Outputs") / "step1_pins_class_mapping" / "pin_class_filtered_unique.csv",
]
pin_class_source_path = next((p for p in pin_class_source_candidates if p.exists()), None)
if pin_class_source_path is not None:
    pin_class_source_df = pd.read_csv(pin_class_source_path, dtype=str).drop_duplicates()
elif "pin_class_df" in globals():
    pin_class_source_df = pin_class_df.copy().drop_duplicates()
else:
    raise FileNotFoundError(f"Cannot find any Step 1 export. Tried: {pin_class_source_candidates}")

print("Pipeline license source:", pipeline_license_source_label)
print("Pipeline pin-class source:", pin_class_source_path)
print("Pipeline skip existing:", PIPELINE_SKIP_EXISTING)
print("Pipeline years:", TARGET_YEARS)

run_logs = []
year_iter = pipeline_tqdm(TARGET_YEARS, desc="Part 4 pipeline", unit="year") if pipeline_tqdm is not None else TARGET_YEARS
for year in year_iter:
    try:
        result = run_one_year_pipeline(year, licenses_full, pin_class_source_df, skip_existing=PIPELINE_SKIP_EXISTING)
        run_logs.append(result)
        if result["status"] == "skipped_existing":
            print(f"[SKIP] year={year} existing step2/step3 outputs found")
        elif result["status"].startswith("skipped_"):
            print(f"[SKIP] year={year} {result['status']}")
        else:
            print(f"[OK] year={year} matched={result['matched_rows']:,} source={result['parcel_source']}")
    except Exception as e:
        run_logs.append({"year": year, "status": "failed", "error": str(e)})
        print(f"[FAIL] year={year}: {e}")

pipeline_log = pd.DataFrame(run_logs)
pipeline_log.to_csv(PROCESS_ALL_LOG, index=False, encoding="utf-8-sig")

print("\nProcess-all log saved:")
print(PROCESS_ALL_LOG)
display(pipeline_log)